## Differences from V1 Baseline (`EffecientNet_V1.ipynb`)

| Component | V1 (Baseline) | V2 (This Notebook) |
|-----------|--------------|-------------------|
| **Model** | `efficientnet_b4` | `tf_efficientnet_b4_ns` (NoisyStudent pretrain) |
| **Image Size** | 380×380 | 512×512 |
| **Loss Function** | `CrossEntropyLoss` | `BiTemperedLogisticLoss` (t1=0.5, t2=1.5) |
| **Learning Rate** | `1e-4` | `3e-4` |
| **Scheduler** | `CosineAnnealingLR` (T_max=10) | `CosineAnnealingWarmRestarts` (T_0=5×steps) |
| **Epochs** | 10 | 15 |
| **CV Training** | Fold 0 only | All 5 folds (ensemble) |
| **Augmentation** | HFlip, VFlip, Rotate(30°), ColorJitter, GaussNoise/Blur | + GridDistortion, CoarseDropout, CutMix, MixUp, MotionBlur, Rotate(45°) |
| **Test-Time Aug** | None | 8× TTA (flips, rotations, crops) |
| **Mixed Precision** | No | Yes (AMP) |


## Results — Kaggle Leaderboard

- **Private Score**: 0.8964
- **Public Score**: 0.8962

---

### Model & Training Hyperparameters

- **Model**: `tf_efficientnet_b4_ns` (NoisyStudent pretrained) *(changed)*
- **Image size**: 512×512 *(changed)*
- **Batch size**: 16
- **Epochs**: 15 *(changed)*
- **Folds trained**: 5 (full ensemble) *(changed)*

**Loss**
- Function: `BiTemperedLogisticLoss` *(changed)*
- t1: 0.5, t2: 1.5 *(changed)*
- Label smoothing: 0.1

**Optimizer** — AdamW
- Learning rate: `3e-4` *(changed)*
- Weight decay: `1e-5`

**Scheduler** — `CosineAnnealingWarmRestarts` *(changed)*
- T_0: 5 × steps per epoch *(changed)*
- eta_min: `1e-6`

**Augmentations (train)**
- RandomResizedCrop (scale 0.7–1.0) *(changed)*
- HorizontalFlip (p=0.5)
- VerticalFlip (p=0.5)
- Rotate(limit=45°, p=0.5) *(changed)*
- ColorJitter (brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1) *(changed)*
- OneOf [GaussNoise, GaussianBlur, MotionBlur] (p=0.3) *(changed)*
- GridDistortion (p=0.2) *(new)*
- CoarseDropout (p=0.3) *(new)*
- CutMix (alpha=1.0, p=0.5) *(new)*
- MixUp (alpha=0.4, p=0.3) *(new)*

**Inference**
- 8× Test-Time Augmentation *(new)*
- 5-fold ensemble averaging *(changed)*
- Automatic Mixed Precision (AMP) *(new)*


In [ ]:
# If you see: "no kernel image is available for execution on the device",
# your preinstalled torch wheel likely doesn't support the current Kaggle GPU.
# This upgrades PyTorch to a CUDA wheel that supports newer GPUs.
!
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Pin albumentations to match the notebook's augmentation argument names.
!pip install -q timm "albumentations==1.3.1"

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import json
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

# Optional AMP (GPU only)
# Prefer modern torch.amp API; fall back for older torch.
try:
    from torch.amp import GradScaler, autocast
except Exception:
    try:
        from torch.cuda.amp import GradScaler, autocast
    except Exception:
        GradScaler, autocast = None, None

# Optional TPU (torch-xla)
# IMPORTANT (PJRT): do NOT call xm.xla_device() before xmp.spawn(),
# otherwise the XLA runtime initializes and spawn will error.
XLA_AVAILABLE = False
xm = None
pl = None
xmp = None
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    import torch_xla.distributed.xla_multiprocessing as xmp
    XLA_AVAILABLE = True
except Exception:
    XLA_AVAILABLE = False


def detect_tpu_via_env() -> bool:
    """Best-effort TPU detection without initializing XLA runtime."""
    pjrt = os.environ.get("PJRT_DEVICE", "").upper()
    if pjrt == "TPU":
        return True

    # Kaggle/Colab commonly expose one or more of these when TPU is enabled.
    for k in ("KAGGLE_TPU", "TPU_NAME", "COLAB_TPU_ADDR"):
        if os.environ.get(k):
            return True

    return False


# Reproducibility
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Prefer GPU if CUDA is available, even if TPU env vars exist.
USE_TPU = bool(XLA_AVAILABLE and detect_tpu_via_env() and (not torch.cuda.is_available()))

if USE_TPU:
    # Kaggle PJRT sometimes exposes a single TPU worker address; limit devices to avoid
    # "Expected 8 worker addresses, got 1" init failures.
    os.environ.setdefault('TPU_NUM_DEVICES', '1')

    # Defer real TPU device initialization to inside xmp.spawn().
    device = torch.device('cpu')
    print("Device: TPU/XLA (deferred init; will initialize inside xmp.spawn)")
    print(f"TPU_NUM_DEVICES={os.environ.get('TPU_NUM_DEVICES')}")
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
class CFG:
    # Reproducibility
    seed = 42

    # Runtime
    use_tpu = USE_TPU

    # Paths — Kaggle notebook (add "Cassava Leaf Disease Classification" data; default mount)
    data_dir = "/kaggle/input/cassava-leaf-disease-classification"
    train_csv = os.path.join(data_dir, "train.csv")
    train_images = os.path.join(data_dir, "train_images")
    test_images = os.path.join(data_dir, "test_images")
    label_map_path = os.path.join(data_dir, "label_num_to_disease_map.json")
    sample_submission = os.path.join(data_dir, "sample_submission.csv")
    output_dir = "/kaggle/working"

    num_folds = 5

    # Model
    model_name = 'tf_efficientnet_b4_ns'   # NoisyStudent — better on noisy labels
    num_classes = 5
    model_save_prefix = 'best_model_v2'    # saved to output_dir

    # Training
    image_size = 512
    batch_size = 16
    num_epochs = 15
    lr = 3e-4
    weight_decay = 1e-5
    T_0 = 5
    eta_min = 1e-6
    use_amp = (not use_tpu) and bool(torch.cuda.is_available())

    # DataLoader
    num_workers = 0 if use_tpu else 2

    # Loss
    bi_tempered_t1 = 0.5
    bi_tempered_t2 = 1.5
    label_smoothing = 0.1

    # Augmentation
    cutmix_alpha = 1.0
    mixup_alpha = 0.4
    cutmix_prob = 0.5
    mixup_prob = 0.3

    # TTA
    num_tta = 8


os.makedirs(CFG.output_dir, exist_ok=True)
seed_everything(CFG.seed)
assert os.path.exists(CFG.data_dir), f"data_dir not found: {CFG.data_dir}"
assert os.path.exists(CFG.train_csv), f"train.csv not found: {CFG.train_csv}"
print(f'Data dir   : {CFG.data_dir}')
print(f'train.csv  : {CFG.train_csv}')
print(f'Output dir : {CFG.output_dir}')
print(f'use_tpu    : {CFG.use_tpu}')
print(f'use_amp    : {CFG.use_amp}')
print(f'num_workers: {CFG.num_workers}')

In [ ]:
# ── Load Data & EDA ────────────────────────────────────────────────────────────
train_df = pd.read_csv(CFG.train_csv)
with open(CFG.label_map_path) as f:
    label_map = json.load(f)

print(f'Total training samples: {len(train_df)}')
print(f'\nClass distribution:')
class_counts = train_df['label'].value_counts().sort_index()
for label, count in class_counts.items():
    print(f'  [{label}] {label_map[str(label)]}: {count} ({count/len(train_df)*100:.1f}%)')

# Plot class distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([label_map[str(i)] for i in range(5)], [class_counts[i] for i in range(5)], color='steelblue')
ax.set_title('Class Distribution (Severe Imbalance: CMD=61.4%, CBB=5.1%)')
ax.set_ylabel('Count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Bi-Tempered Logistic Loss ──────────────────────────────────────────────────
# Reference: Amid et al. (2019) — handles noisy labels by using tempered softmax.
# t1 < 1 makes loss bounded for mislabeled samples; t2 > 1 makes tail of softmax heavier.

def log_t(u, t):
    """Compute log_t for a given t (tempered log)."""
    if t == 1.0:
        return torch.log(u)
    return (u.pow(1.0 - t) - 1.0) / (1.0 - t)


def exp_t(u, t):
    """Compute exp_t for a given t (tempered exp)."""
    if t == 1.0:
        return torch.exp(u)
    return torch.relu(1.0 + (1.0 - t) * u).pow(1.0 / (1.0 - t))


def compute_normalization(activations, t, num_iters=5):
    """Return the normalization value for each example (tempered softmax denominator)."""
    mu = activations.max(dim=-1, keepdim=True).values
    effective_dim = torch.tensor(activations.shape[-1], dtype=activations.dtype, device=activations.device)
    z = mu + ((effective_dim * (1 - t)).pow(1.0 / (2.0 - t))) / (2.0 - t)
    for _ in range(num_iters):
        z = mu + (exp_t(activations - z, t)).sum(dim=-1, keepdim=True)
    return z


def tempered_softmax(activations, t):
    """Tempered softmax function."""
    if t == 1.0:
        return F.softmax(activations, dim=-1)
    normalization = compute_normalization(activations, t)
    return exp_t(activations - normalization, t)


class BiTemperedLogisticLoss(nn.Module):
    """
    Bi-Tempered Logistic Loss.
    t1 (< 1): Controls robustness to noisy labels (bounded loss for outliers).
    t2 (> 1): Controls tail-heaviness of the probability distribution.
    """
    def __init__(self, t1=0.5, t2=1.5, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.t1 = t1
        self.t2 = t2
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, labels):
        """
        logits: (B, C) raw model outputs
        labels: (B,) integer class indices OR (B, C) soft label tensor
        """
        num_classes = logits.shape[-1]

        # Convert integer labels to one-hot if needed
        if labels.dim() == 1:
            labels_onehot = F.one_hot(labels.long(), num_classes).float()
        else:
            labels_onehot = labels.float()

        # Apply label smoothing
        if self.label_smoothing > 0:
            labels_onehot = labels_onehot * (1 - self.label_smoothing) + \
                            self.label_smoothing / num_classes

        probabilities = tempered_softmax(logits, self.t2)

        loss_values = (
            labels_onehot * log_t(labels_onehot + 1e-10, self.t1)
            - labels_onehot * log_t(probabilities, self.t1)
            - (1.0 / (2.0 - self.t1)) * (labels_onehot.pow(2.0 - self.t1) - probabilities.pow(2.0 - self.t1))
        ).sum(dim=-1)

        if self.reduction == 'mean':
            return loss_values.mean()
        elif self.reduction == 'sum':
            return loss_values.sum()
        return loss_values

In [ ]:
# ── Augmentations ─────────────────────────────────────────────────────────────
_SIZE = (CFG.image_size, CFG.image_size)   # tuple required by albumentations >= 1.4

def get_train_transforms():
    return A.Compose([
        A.RandomResizedCrop(_SIZE, scale=(0.7, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=45, p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=7),
        ], p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        A.CoarseDropout(
            max_holes=8, max_height=CFG.image_size // 16, max_width=CFG.image_size // 16,
            min_holes=1, fill_value=0, p=0.3
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_valid_transforms():
    return A.Compose([
        A.Resize(CFG.image_size, CFG.image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


# TTA transforms: 8 deterministic augmentation variants
def get_tta_transforms():
    _norm = [A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()]
    _resize = A.Resize(CFG.image_size, CFG.image_size)
    _crop   = A.RandomResizedCrop(_SIZE, scale=(0.85, 1.0), p=1.0)
    return [
        A.Compose([_resize,                                              *_norm]),
        A.Compose([_resize, A.HorizontalFlip(p=1.0),                    *_norm]),
        A.Compose([_resize, A.VerticalFlip(p=1.0),                      *_norm]),
        A.Compose([_resize, A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0), *_norm]),
        A.Compose([_resize, A.Rotate(limit=90, p=1.0),                  *_norm]),
        A.Compose([_resize, A.Rotate(limit=90, p=1.0), A.HorizontalFlip(p=1.0), *_norm]),
        A.Compose([_crop,                                                *_norm]),
        A.Compose([_crop,   A.HorizontalFlip(p=1.0),                    *_norm]),
    ]

In [ ]:
# ── CutMix & MixUp ────────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    """Random bounding box for CutMix."""
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix(images, labels, alpha=1.0):
    """Apply CutMix augmentation. Returns mixed images and label pair."""
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    rand_index = torch.randperm(batch_size, device=images.device)

    labels_a = labels
    labels_b = labels[rand_index]

    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images[:, :, x1:x2, y1:y2] = images[rand_index, :, x1:x2, y1:y2]

    # Adjust lam to actual area ratio
    lam = 1 - ((x2 - x1) * (y2 - y1)) / (images.size(-1) * images.size(-2))
    return images, labels_a, labels_b, lam


def mixup(images, labels, alpha=0.4):
    """Apply MixUp augmentation. Returns mixed images and label pair."""
    lam = np.random.beta(alpha, alpha)
    batch_size = images.size(0)
    rand_index = torch.randperm(batch_size, device=images.device)

    labels_a = labels
    labels_b = labels[rand_index]

    mixed_images = lam * images + (1 - lam) * images[rand_index]
    return mixed_images, labels_a, labels_b, lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    """Loss for CutMix/MixUp: weighted combination."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_id'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        label = row.get('label', -1)
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(pretrained=True):
    """
    EfficientNet-B4 NoisyStudent — pretrained with noisy student self-training
    on ImageNet. Outperforms standard B4 on real-world noisy datasets.
    """
    model = timm.create_model(CFG.model_name, pretrained=pretrained)
    n_features = model.classifier.in_features
    model.classifier = nn.Linear(n_features, CFG.num_classes)
    return model.to(device)

In [ ]:
# ── Training & Validation Functions ───────────────────────────────────────────
from contextlib import nullcontext


def _wrap_loader(loader):
    # On TPU, MpDeviceLoader moves batches to the XLA device efficiently.
    return pl.MpDeviceLoader(loader, device) if CFG.use_tpu else loader


def _optimizer_step(optimizer):
    if CFG.use_tpu:
        xm.optimizer_step(optimizer, barrier=True)
        xm.mark_step()
    else:
        optimizer.step()


def _autocast_ctx():
    """Return the correct autocast context for this torch version."""
    if (not CFG.use_amp) or CFG.use_tpu or (autocast is None) or (device.type != 'cuda'):
        return nullcontext()
    try:
        return autocast(device_type='cuda')
    except TypeError:
        return autocast()


def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    loader = _wrap_loader(loader)

    for images, labels in loader:
        if not CFG.use_tpu:
            images = images.to(device)
            labels = labels.to(device)

        # Randomly apply CutMix or MixUp
        r = random.random()
        use_mix = False
        if r < CFG.cutmix_prob:
            images, labels_a, labels_b, lam = cutmix(images, labels, alpha=CFG.cutmix_alpha)
            use_mix = True
        elif r < CFG.cutmix_prob + CFG.mixup_prob:
            images, labels_a, labels_b, lam = mixup(images, labels, alpha=CFG.mixup_alpha)
            use_mix = True

        optimizer.zero_grad(set_to_none=True)

        # AMP is GPU-only in this notebook; TPU runs fp32 by default.
        if CFG.use_amp and (scaler is not None):
            with _autocast_ctx():
                outputs = model(images)
                if use_mix:
                    loss = mixed_criterion(criterion, outputs, labels_a, labels_b, lam)
                else:
                    loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            if use_mix:
                loss = mixed_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
            loss.backward()
            _optimizer_step(optimizer)

        scheduler.step()

        # Accuracy (using original labels for tracking, not mixed)
        preds = outputs.argmax(dim=1)
        orig_labels = labels_a if use_mix else labels
        total_correct += (preds == orig_labels).sum().item()
        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples



def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    loader = _wrap_loader(loader)

    with torch.no_grad():
        for images, labels in loader:
            if not CFG.use_tpu:
                images = images.to(device)
                labels = labels.to(device)

            with _autocast_ctx():
                outputs = model(images)
                loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)

            # For TPU, preds/labels are XLA tensors; .cpu() transfers them.
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total_samples, total_correct / total_samples, all_preds, all_labels

In [ ]:
# ── 5-Fold Cross-Validation Training ──────────────────────────────────────────
import pickle

train_img_dir = CFG.train_images
skf = StratifiedKFold(n_splits=CFG.num_folds, shuffle=True, random_state=CFG.seed)

fold_scores = []
fold_history = []  # For plotting

criterion = BiTemperedLogisticLoss(
    t1=CFG.bi_tempered_t1,
    t2=CFG.bi_tempered_t2,
    label_smoothing=CFG.label_smoothing
)


def _run_cv_single_process():
    global fold_scores, fold_history

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
        print(f'\n{"="*60}')
        print(f' FOLD {fold+1} / {CFG.num_folds}')
        print(f'{"="*60}')

        train_fold = train_df.iloc[train_idx]
        val_fold = train_df.iloc[val_idx]

        train_dataset = CassavaDataset(train_fold, train_img_dir, transform=get_train_transforms())
        val_dataset = CassavaDataset(val_fold, train_img_dir, transform=get_valid_transforms())

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.batch_size,
            shuffle=True,
            num_workers=CFG.num_workers,
            pin_memory=(not CFG.use_tpu),
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=(not CFG.use_tpu),
        )

        model = build_model(pretrained=True)
        optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
        scheduler = CosineAnnealingWarmRestarts(
            optimizer, T_0=CFG.T_0 * len(train_loader), eta_min=CFG.eta_min
        )
        if GradScaler is None:
            scaler = None
        else:
            try:
                scaler = GradScaler('cuda', enabled=CFG.use_amp)
            except TypeError:
                scaler = GradScaler(enabled=CFG.use_amp)

        best_val_acc = 0.0
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

        for epoch in range(1, CFG.num_epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, scheduler)
            val_loss, val_acc, val_preds, val_labels = validate_one_epoch(model, val_loader, criterion)

            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            print(f'  Epoch {epoch:02d}/{CFG.num_epochs} | '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}', end='')

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                save_path = os.path.join(CFG.output_dir, f'{CFG.model_save_prefix}_fold{fold}.pth')
                torch.save(model.state_dict(), save_path)
                print(f'  -> Saved ({save_path})')
            else:
                print()

        print(f'\n  Best Val Accuracy for Fold {fold}: {best_val_acc:.4f}')
        fold_scores.append(best_val_acc)
        fold_history.append(history)

    print(f'\n{"="*60}')
    print(f'CV RESULTS: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
    print(f'Per-fold: {[f"{s:.4f}" for s in fold_scores]}')
    print(f'{"="*60}')


TPU_CV_RESULTS_PATH = os.path.join(CFG.output_dir, 'tpu_cv_results.pkl')


def tpu_cv_mp_fn(index):
    """TPU worker function (must be top-level to be picklable under PJRT)."""
    global device
    device = xm.xla_device()
    seed_everything(CFG.seed + xm.get_ordinal())

    # Only master prints/saves
    is_master = xm.is_master_ordinal()

    local_fold_scores = []
    local_fold_history = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
        if is_master:
            xm.master_print(f"\n{'='*60}\n FOLD {fold+1} / {CFG.num_folds}\n{'='*60}")

        train_fold = train_df.iloc[train_idx]
        val_fold = train_df.iloc[val_idx]

        train_dataset = CassavaDataset(train_fold, train_img_dir, transform=get_train_transforms())
        val_dataset = CassavaDataset(val_fold, train_img_dir, transform=get_valid_transforms())

        # Shard training across TPU cores
        train_sampler = torch.utils.data.DistributedSampler(
            train_dataset,
            num_replicas=xm.xrt_world_size(),
            rank=xm.get_ordinal(),
            shuffle=True,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.batch_size,
            sampler=train_sampler,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=False,
            drop_last=True,
        )

        # Validate on master only (simpler than cross-core metric reduction)
        if is_master:
            val_loader = DataLoader(
                val_dataset,
                batch_size=CFG.batch_size,
                shuffle=False,
                num_workers=CFG.num_workers,
                pin_memory=False,
            )
        else:
            val_loader = None

        model = build_model(pretrained=True)
        optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
        scheduler = CosineAnnealingWarmRestarts(
            optimizer, T_0=CFG.T_0 * len(train_loader), eta_min=CFG.eta_min
        )

        best_val_acc = 0.0
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

        for epoch in range(1, CFG.num_epochs + 1):
            train_sampler.set_epoch(epoch)
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, None, scheduler)

            if is_master:
                val_loss, val_acc, _, _ = validate_one_epoch(model, val_loader, criterion)

                history['train_loss'].append(train_loss)
                history['train_acc'].append(train_acc)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)

                xm.master_print(
                    f"  Epoch {epoch:02d}/{CFG.num_epochs} | "
                    f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                    f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}",
                    end='',
                )

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    save_path = os.path.join(CFG.output_dir, f'{CFG.model_save_prefix}_fold{fold}.pth')
                    xm.save(model.state_dict(), save_path)
                    xm.master_print(f'  -> Saved ({save_path})')
                else:
                    xm.master_print('')

            xm.rendezvous(f"fold_{fold}_epoch_{epoch}")

        if is_master:
            xm.master_print(f"\n  Best Val Accuracy for Fold {fold}: {best_val_acc:.4f}")
            local_fold_scores.append(best_val_acc)
            local_fold_history.append(history)

    if is_master:
        xm.master_print(f"\n{'='*60}")
        xm.master_print(f"CV RESULTS: {np.mean(local_fold_scores):.4f} +/- {np.std(local_fold_scores):.4f}")
        xm.master_print(f"Per-fold: {[f'{s:.4f}' for s in local_fold_scores]}")
        xm.master_print(f"{'='*60}")

        with open(TPU_CV_RESULTS_PATH, 'wb') as f:
            pickle.dump({'fold_scores': local_fold_scores, 'fold_history': local_fold_history}, f)



def _run_cv_tpu_multiprocess():
    """Run CV on TPU across all XLA cores."""
    # PJRT-based torch-xla (current Kaggle) expects nprocs=None to use all devices.
    # 'spawn' is more reliable than 'fork' in notebook environments.
    
    

    # Load back results into this (parent) process
    with open(TPU_CV_RESULTS_PATH, 'rb') as f:
        obj = pickle.load(f)
    fold_scores[:] = obj['fold_scores']
    fold_history[:] = obj['fold_history']


if CFG.use_tpu:
    _run_cv_tpu_multiprocess()
else:
    _run_cv_single_process()

In [ ]:
# ── Plot Training Curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(CFG.num_folds, 2, figsize=(14, 4 * CFG.num_folds))

for fold in range(CFG.num_folds):
    h = fold_history[fold]
    epochs = range(1, len(h['train_loss']) + 1)

    axes[fold, 0].plot(epochs, h['train_loss'], label='Train')
    axes[fold, 0].plot(epochs, h['val_loss'], label='Val')
    axes[fold, 0].set_title(f'Fold {fold+1} — Loss')
    axes[fold, 0].set_xlabel('Epoch')
    axes[fold, 0].legend()

    axes[fold, 1].plot(epochs, h['train_acc'], label='Train')
    axes[fold, 1].plot(epochs, h['val_acc'], label='Val')
    axes[fold, 1].set_title(f'Fold {fold+1} — Accuracy (Best: {fold_scores[fold]:.4f})')
    axes[fold, 1].set_xlabel('Epoch')
    axes[fold, 1].legend()

plt.suptitle(f'V2 Training: CV Mean = {np.mean(fold_scores):.4f}', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, 'training_curves_v2.png'), dpi=100, bbox_inches='tight')
plt.show()

## Inference — 5-Fold Ensemble + 8× TTA

Load all 5 fold checkpoints and predict on the test set using 8 TTA passes per fold (40 total predictions averaged).

In [ ]:
# ── TTA Inference Helper ───────────────────────────────────────────────────────
class TTADataset(Dataset):
    """Dataset that applies one specific TTA transform."""
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.img_dir, img_id)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)['image']
        return image


def predict_with_tta(model, df, img_dir):
    """Run all TTA transforms and return averaged softmax probabilities."""
    model.eval()
    tta_transforms = get_tta_transforms()
    all_probs = []

    for tta_t in tta_transforms:
        dataset = TTADataset(df, img_dir, tta_t)
        loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=False,
                            num_workers=CFG.num_workers, pin_memory=True)
        batch_probs = []
        with torch.no_grad():
            for images in loader:
                images = images.to(device)
                with _autocast_ctx():
                    logits = model(images)
                probs = F.softmax(logits, dim=1)
                batch_probs.append(probs.cpu().numpy())
        all_probs.append(np.concatenate(batch_probs, axis=0))

    # Average over all TTA transforms
    return np.mean(all_probs, axis=0)

In [ ]:
# ── Load Test Data & Run Ensemble Inference ────────────────────────────────────
test_img_dir = CFG.test_images
test_df = pd.DataFrame({'image_id': os.listdir(test_img_dir)})
print(f'Test images: {len(test_df)}')

all_fold_probs = []

for fold in range(CFG.num_folds):
    checkpoint_path = os.path.join(CFG.output_dir, f'{CFG.model_save_prefix}_fold{fold}.pth')
    print(f'Loading fold {fold} checkpoint: {checkpoint_path}')

    model = build_model(pretrained=False)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)

    fold_probs = predict_with_tta(model, test_df, test_img_dir)
    all_fold_probs.append(fold_probs)
    print(f'  Fold {fold} TTA done. Shape: {fold_probs.shape}')

# Ensemble: average across all folds
ensemble_probs = np.mean(all_fold_probs, axis=0)   # (N, 5)
predictions = np.argmax(ensemble_probs, axis=1)
print(f'\nFinal ensemble prediction shape: {ensemble_probs.shape}')
print(f'Prediction distribution: {dict(zip(*np.unique(predictions, return_counts=True)))}')

In [ ]:
# ── Generate Submission ────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'image_id': test_df['image_id'],
    'label': predictions
})
submission_path = os.path.join(CFG.output_dir, 'submission_v2.csv')
submission.to_csv(submission_path, index=False)
print(f'Saved: {submission_path}')
submission.head(10)

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('V2 TRAINING SUMMARY')
print('=' * 60)
print(f'Model          : {CFG.model_name}')
print(f'Image size     : {CFG.image_size}×{CFG.image_size}')
print(f'Epochs per fold: {CFG.num_epochs}')
print(f'Loss           : BiTemperedLogisticLoss (t1={CFG.bi_tempered_t1}, t2={CFG.bi_tempered_t2})')
print(f'Augmentations  : CutMix + MixUp + GridDistortion + CoarseDropout')
print(f'TTA passes     : {CFG.num_tta}')
print('-' * 60)
for fold, score in enumerate(fold_scores):
    print(f'  Fold {fold}: {score:.4f}')
print(f'  Mean  : {np.mean(fold_scores):.4f} \u00b1 {np.std(fold_scores):.4f}')
print('=' * 60)
print(f'Checkpoints    : {CFG.output_dir}/{CFG.model_save_prefix}_fold{{0..{CFG.num_folds-1}}}.pth')
print(f'Submission     : {CFG.output_dir}/submission_v2.csv')